> **Portfolio summary:** Semantic search over financial news using sentence embeddings, with a Gradio interface.
>
> [Project README](README.md) · [Back to the portfolio](../README.md)

## Reflective Introduction

In today's data-rich world, simply finding information is no longer enough; we need to understand its meaning and context. This project delves into building a semantic search tool for financial news, moving beyond keyword matching to uncover true relevance. By leveraging advanced natural language processing techniques, specifically sentence embeddings, we aim to transform how users interact with financial data, making insights more accessible and actionable. This exploration highlights the power of AI in bridging the gap between raw text and meaningful understanding.

In [ ]:
import pandas as pd

file_path = '/content/financial_news.csv'
df = pd.read_csv(file_path)

print(f"Dataset loaded from: {file_path}")

Dataset loaded from: /content/financial_news.csv

### First 5 rows of the dataset:

In [ ]:
display(df.head())

,text,label
0,Here are Thursday's biggest analyst calls: App...,0
1,Buy Las Vegas Sands as travel to Singapore bui...,0
2,"Piper Sandler downgrades DocuSign to sell, cit...",0
3,"Analysts react to Tesla's latest earnings, bre...",0
4,Netflix and its peers are set for a ‘return to...,0


###Dataset shape (rows, columns):

In [ ]:
print(df.shape)

(16990, 2)

###Dataset information (column names, non-null counts, data types):

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16990 entries, 0 to 16989
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    16990 non-null  object
 1   label   16990 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 265.6+ KB

###### Displaying the DataFrame with the new `URL` column and updated `text` column:

In [ ]:
import re

# Define a regex pattern to find URLs starting with http or https at the end of the string
url_pattern = r'(https?://[^\s]+)$'

# Extract URLs into a new 'URL' column
df['URL'] = df['text'].str.extract(url_pattern, flags=re.IGNORECASE)

# Remove URLs from the 'text' column
df['text'] = df['text'].str.replace(url_pattern, '', regex=True)

print("URLs extracted and removed from 'text' column. New 'URL' column created.")

URLs extracted and removed from 'text' column. New 'URL' column created.

In [ ]:
df.head()

,text,label,URL
0,Here are Thursday's biggest analyst calls: App...,0,https://t.co/QPN8Gwl7Uh
1,Buy Las Vegas Sands as travel to Singapore bui...,0,https://t.co/fLS2w57iCz
2,"Piper Sandler downgrades DocuSign to sell, cit...",0,https://t.co/1EmtywmYpr
3,"Analysts react to Tesla's latest earnings, bre...",0,https://t.co/kwhoE6W06u
4,Netflix and its peers are set for a ‘return to...,0,https://t.co/jPpdl0D9s4


# Objective
Build a semantic search tool for the financial news dataset (stored in the DataFrame `df`) using `sentence-transformers` for generating embeddings and `Gradio` for creating an interactive user interface. The tool should allow users to input a query and retrieve the top 5 most semantically similar news articles from the `text` column, displaying their content, URLs, and similarity scores.

## Generate Sentence Embeddings

### Subtask:
Install the `sentence-transformers` library and load a pre-trained model (e.g., 'all-MiniLM-L6-v2'). Then, create sentence embeddings for the cleaned `text` column of the DataFrame.


In [ ]:
pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

# Load a pre-trained model
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Sentence-BERT model 'all-MiniLM-L6-v2' loaded.")

# Generate embeddings for the 'text' column
sentence_embeddings = model.encode(df['text'].tolist())
print(f"Embeddings generated for {len(sentence_embeddings)} sentences. Shape: {sentence_embeddings.shape}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def semantic_search(query: str, top_n: int = 5):
    """
    Performs semantic search on the financial news dataset.

    Args:
        query (str): The search query.
        top_n (int): The number of top similar articles to retrieve.

    Returns:
        pd.DataFrame: A DataFrame containing the top_n most similar articles
                      with their text, URL, and similarity score.
    """
    # Encode the query
    query_embedding = model.encode([query])

    # Compute cosine similarity between query and all article embeddings
    similarities = cosine_similarity(query_embedding, sentence_embeddings)[0]

    # Get the indices of the top_n most similar articles
    top_n_indices = similarities.argsort()[-top_n:][::-1]

    # Prepare results
    results = []
    for i in top_n_indices:
        results.append({
            'Text': df.loc[i, 'text'],
            'URL': df.loc[i, 'URL'] if pd.notna(df.loc[i, 'URL']) else 'N/A',
            'Similarity Score': similarities[i]
        })

    return pd.DataFrame(results)

print("Semantic search function 'semantic_search' defined.")

## Build Gradio Interface

Construct a Gradio application for the semantic search tool.

In [ ]:
pip install gradio

In [ ]:
import gradio as gr

# Create the Gradio interface
iface = gr.Interface(
    fn=semantic_search,
    inputs=gr.Textbox(lines=2, label='Enter your search query:'),
    outputs=gr.Dataframe(
        headers=["Text", "URL", "Similarity Score"],
        row_count=5,
        col_count=3,
        label="Top 5 Similar Articles"
    ),
    title="Financial News Semantic Search",
    description="Enter a query to find the top 5 most semantically similar financial news articles."
)

# Launch the interface
print("Launching Gradio interface...")
iface.launch(share=True)